In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
import yaml

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score

import optuna
import mne

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import sys
from pathlib import Path

# Находим корень проекта (поднимаемся на уровень выше папки notebooks)
root = Path.cwd().parent
sys.path.append(str(root))
sys.path.append(r"/trinity/home/asma.benachour/BrainBERT")

from utils.TFRDataset import TFRDataset

from utils.normalisation import normalize_tfr_robust
from utils.AlexNet import AlexNetTFR
from lib.optuna_objective_makers import *
from utils.optuna_constraints import slope_constraint

from collections import Counter

from optuna.trial import TrialState
from utils.optuna_study_analyzers import pareto_front, feasible_trials_less_zero
import gc

from utils.train_eval_helpers import train_one_epoch, eval_one_epoch_f1_macro

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
with open('../patients.yaml') as f:
    test = yaml.safe_load(f)
test
pat_config = test['default']
# time_frequency_file = pat_config['time_frequency_file'] 
time_frequency_file = "../specs_with_car/tfr_s11.fif"
min_f, max_f, min_t, max_t = pat_config['min_f'], pat_config['max_f'], pat_config['min_t'], pat_config['max_t']

tfr = mne.time_frequency.read_tfrs(time_frequency_file)[0]
y = np.where(tfr.events[:, 2] == 9, 1, 0)
X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, min_f:max_f, min_t:max_t]
# X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, :-50, :]
del tfr
gc.collect() # Теперь память гарантировано свободна

Reading ../specs_with_car/tfr_s11.fif ...


/tmp/ipykernel_1737409/421300773.py:9: RuntimeWarning: This filename (../specs_with_car/tfr_s11.fif) does not conform to MNE naming conventions. All tfr files should end with -tfr.h5 or _tfr.h5
  tfr = mne.time_frequency.read_tfrs(time_frequency_file)[0]


Not setting metadata


831

In [5]:
cv=False
max_epochs = 100
in_channels = 7 
num_classes = 2
channels = X.shape[1]
seed = 42
test_size = 0.2
n_trials_optuna_whole = 50
n_startup_trials, n_warmup_steps = 5, 5
patience=6
constraints_func = slope_constraint
cv_aggregate = "median"
sampler = optuna.samplers.NSGAIISampler(
    seed=seed,
    constraints_func=constraints_func,   # <-- сюда
)

study = optuna.create_study(
    # макс метрику качества, мин метрику ошибки
    directions=["maximize", "minimize"],
    # sampler=optuna.samplers.TPESampler(), для сингл обджектива
    sampler=sampler
    # pruner=optuna.pruners.MedianPruner(n_startup_trials=n_startup_trials, n_warmup_steps=n_warmup_steps) # может вызвать ошибки, так как прунер настроен на одиночную метрику
)


/trinity/home/t.samsonov/miniconda3/envs/brainbert/lib/python3.10/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``constraints_func`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-03-17 14:29:23,932] A new study created in memory with name: no-name-8d1d1b3a-380c-44c4-96a5-2e1edcac090e


In [6]:
objective = make_objective_engine(
    X=X, y=y,
    make_splits_fn=make_splits_fn_factory(test_size=test_size, seed=seed, cv=cv),
    run_fold_fn=run_fold_fn_factory(
        ModelCls=AlexNetTFR, device=device, 
        max_epochs=max_epochs, patience=patience,
        TFRDataset=TFRDataset, DataLoader=DataLoader,
        train_one_epoch=train_one_epoch, eval_one_epoch_f1_macro=eval_one_epoch_f1_macro,
        loss_metric=loss_slope
    ),
    aggregate_mode=cv_aggregate,
    params_fn=params_fn_factory(
        in_channels=in_channels, num_classes=num_classes,
    ),
    objectives_fn=objectives_fn,
    attrs_fn=attrs_fn,
)


TypeError: run_fold_fn_factory() got an unexpected keyword argument 'torch'

In [ ]:
study.optimize(objective, n_trials=n_trials_optuna_whole, show_progress_bar=True)